# A1 - Decoder a segmenti per il possesso (DFL)

Per ogni frame (25 fps) assegniamo **H** (casa in possesso), **A** (ospiti) o **D** (palla morta).
La classifica usa **spellF1**: ogni possesso (run massimale di H o di A) è un oggetto, accoppiato a quelli veri per IoU temporale; la F1 è mediata sulle soglie 0.3-0.7.
Uno sfarfallio di pochi frame dentro un possesso lo spezza in due pezzi che spesso non raggiungono più l'IoU richiesto: un falso positivo e un mancato per un errore minuscolo.

Questo notebook sostituisce il voto a maggioranza su finestra del baseline con una **decodifica esatta a segmenti** e la misura sulle 7 partite DFL (Bassek et al. 2025, CC BY 4.0: partite diverse da quella della challenge, tuning permesso).
Tutta la logica è in `pipeline/decoder.py`; qui si chiamano solo le funzioni.

**Il nostro contributo.** Dalla ricerca bibliografica non risulta alcun lavoro pubblicato che ottimizzi esplicitamente una *metrica a segmenti* per il possesso: i lavori esistenti classificano frame o eventi. Qui il costo per segmento della decodifica è scelto proprio perché corrisponde all'unità contata da spellF1.

## 1. Setup
Funziona in locale e su Kaggle. Su Kaggle: caricare la repo come dataset `hackaton-barcellona-code` (o clonarla in `/kaggle/working/Hackaton_Barcellona`), oppure impostare `MTAH_REPO`.
I dati DFL convertiti (`DFL-MAT-*.npz`, prodotti da `pipeline/convert_dfl.py`) vanno in un dataset Kaggle qualsiasi sotto `/kaggle/input`, oppure si imposta `DFL_CONVERTED`. Gli output vanno in `/kaggle/working/dfl_outputs` (o `DFL_OUT`).

In [ ]:
import os, sys
REPO = os.environ.get("MTAH_REPO") or next((p for p in ["/kaggle/input/hackaton-barcellona-code", "/kaggle/working/Hackaton_Barcellona", os.path.abspath("..")] if os.path.isdir(os.path.join(p, "pipeline"))), None)
assert REPO, "Imposta MTAH_REPO alla cartella della repo"
sys.path[:0] = [REPO, os.path.join(REPO, "pipeline"), os.path.join(REPO, "starter"), os.path.join(REPO, "evaluation")]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import decoder, dfl_paths

paths = dfl_paths.match_paths()
print("partite DFL convertite:", len(paths), "in", dfl_paths.converted_dir())
print("output in:", dfl_paths.out_dir())
import dfl_sim
print("livello della simulazione:", dfl_sim.level())

Il livello della simulazione (`easy`, `medium`, `hard`) si sceglie con `$DFL_SIM` prima di lanciare il notebook; il default è `medium`. Vedi `pipeline/dfl_sim.py` e `ARCHITETTURA.md`.

## 2. Cosa fa il decoder

`decoder.decode(logp, half, switch_cost, min_dur, context_cost)` sceglie l'intera sequenza di etichette massimizzando

$$\sum_f \log p_f(y_f) \;-\; \sum_{\text{cambi}} \big[\mathrm{switch}(y_{f-1},y_f) + \mathrm{context}_f(y_{f-1},y_f)\big]$$

con il vincolo che ogni run dell'etichetta $k$ duri almeno `min_dur[k]` frame.

* **Penalità per segmento**: ogni cambio costa `switch_cost` (in nat, le unità di `logp`). Un'inversione breve viene accettata solo se l'evidenza sommata su tutta la sua durata paga i *due* cambi che crea. È esattamente l'unità di spellF1: un possesso in più o in meno.
* **Durata minima**: vieta i run cortissimi, che non possono mai essere accoppiati (IoU ~ 0). Per D usiamo almeno 25 frame: Lang et al. (2022) filtrano la probabilità di palla morta con una mediana di ~3.6 s, e nei dati DFL le interruzioni <= 1 s sono quasi inesistenti (2 su 711).
* **Esatta**: programmazione dinamica sugli stati (etichetta, lunghezza del run troncata a `min_dur`), cioè un modello semi-Markov con durata minima (Viterbi a segmenti). La catena delle lunghezze è uno shift deterministico e si collassa con somme cumulative: O(F·K²), circa 0.5 s per 150k frame. Nessuna beam search (approssimata e senza precedenti in letteratura, scartata). I test (`python pipeline/test_decoder.py`) confrontano il risultato con la ricerca esaustiva su sequenze corte.
* **Tempi**: ogni tempo (`half`) è decodificato a parte: il cambio all'intervallo è gratis e nessun run lo attraversa.
* **Contesto**: `context_cost` (F,3,3) rende il costo del cambio dipendente dal frame. PathCRF (Kim et al. 2026, KDD, stesso dataset DFL) mostra che la decodifica strutturata (CRF + Viterbi con transizioni mascherate e *dinamiche*) porta l'F1 sugli eventi da 65.3 a 75.7 e riduce i cambi spuri (1961 eventi predetti contro 2135 con transizioni statiche; 1831 veri).

**Perché sostituisce il voto a finestra.** Il voto +/-50 rende rari gli sfarfallii ma non li elimina (oscilla ovunque la maggioranza locale oscilla) e sposta ogni confine vero fino a mezza finestra. Il decoder ragiona sull'intera sequenza: decide *quanti* segmenti servono e li mette dove l'evidenza cambia.

Esempio sintetico: un possesso H-A-D-H con il 25% dei frame etichettati a caso.

In [ ]:
rng = np.random.default_rng(0)
truth = np.array(["H"] * 200 + ["A"] * 150 + ["D"] * 60 + ["H"] * 200)
noisy = truth.copy()
flip = rng.random(len(truth)) < 0.25
noisy[flip] = rng.choice(decoder.LABELS, flip.sum())
q = np.full((len(noisy), 3), 0.1)
q[np.arange(len(noisy)), [list(decoder.LABELS).index(x) for x in noisy]] = 0.8
lab = decoder.decode(np.log(q), switch_cost=10, min_dur=25)
print("run nell'input rumoroso:", len(decoder.run_lengths(noisy)))
print("run dopo il decoder:   ", [(int(a), int(z), str(k)) for a, z, k in decoder.run_lengths(lab)])

## 3. Evidenza per frame (`frame_logp`)

`logp` (F,3) è costruito in modo agnostico rispetto alla fonte:

* **giocatore più vicino** (`dfl_control.nearest`): la sua squadra, con confidenza 0.75 a 0 m che sfuma a 0.5 con la distanza (scala 3 m);
* **tocchi** (`dfl_control.touches`, soglia 3 m/s): evidenza forte (0.95) per la squadra che tocca, più l'uscita della **regola del tocco** (chi ha toccato per ultimo) come evidenza moderata (0.8);
* **palla morta**: `p_dead` dal file `<match>_deadball.npz` del notebook B1 se esiste, altrimenti il fallback del baseline (palla mancante nella palla "metrica_like": p_dead 0.9, altrimenti 0.02);
* **B2**: `p_home` da `<match>_rule.npz` se esiste, sommato in log-odds.

Dove la palla manca **non c'è evidenza su H/A** (uniforme fra H e A): non è un'evidenza di D. Tutte le probabilità sono clippate a [1e-4, 1-1e-4].

## 4. Valutazione

Condizioni di `pipeline/dfl_sim.py`, la simulazione comune a B1, B2, C1 e A1 (seed 0 e 1; solo il seed 0 quando si usano i file di B1, B2 o C1, che sono calcolati su quello): una sola palla, quella fornita, sfocata (12 frame) e mancante anche a gioco vivo; giocatori con rumore, buchi e tracce che cambiano slot.

Metodi confrontati (spellF1 e macroF1 con `dfl_bench.score`, cioè la metrica ufficiale):
1. **baseline nearest +/-50**: giocatore più vicino sulla palla Metrica + voto +/-50 frame;
2. **touch rule**: regola del tocco di `dfl_control`/`dfl_method4`, senza decoder;
3. **decoder**: `switch_cost` in {4, 8, 16, 32, 64, 128} x `min_dur` (H/A, D) in {(12,25), (25,25), (50,25), (25,50)}, scelti in **leave-one-match-out** (si sceglie sui 6, si misura sul 7°);
4. varianti con gli stessi parametri del decoder:
   * **+refine**: confini H<->A spostati entro +/-12 frame verso un tocco della squadra entrante (EVALUATION.md: 1 frame di ritardo è gratis, 8 frame costano ~0.11, 1 s ~0.35);
   * **+touch_ctx**: cambio verso chi tocca il 50% più economico nel frame del tocco;
   * **+pressure_ctx** (costo dinamico alla PathCRF): cambio H<->A il 50% più economico se un giocatore della squadra entrante (l'avversario di chi ha palla) è entro 1.5 m dalla palla, il 50% più caro se nessuno è entro 5 m. Da confrontare con il costo costante (`decoder`).

Ci vogliono alcuni minuti (7 partite x 2 seed x 24 combinazioni). Salva `<match>_decoded.npz` (seed 0, parametri scelti sulle altre partite) secondo il contratto di `dfl_paths`.

In [ ]:
res = decoder.evaluate_dfl()

In [ ]:
res["summary"]

### Come leggere i risultati
* Ogni riga è un metodo; `mean`/`min`/`max` sono sulle 7 partite (ogni partita mediata sui 2 seed). Per il decoder ogni partita è misurata con i parametri scelti **sulle altre 6**: è una stima onesta, non in-sample.
* Il criterio è **spellF1**; macroF1 è il primo spareggio. Differenze sotto ~0.04 di spellF1 sono rumore su una singola partita (EVALUATION.md): conta il guadagno medio e se migliora anche il minimo per partita.
* `+pressure_ctx` contro `decoder` misura il valore del costo dinamico rispetto al costo costante, a parità di parametri.
* Se le combinazioni scelte nei fold sono quasi sempre le stesse, la scelta è stabile e si può usare così com'è sulla partita della challenge.

In [ ]:
print("fonti:", res["sources"])
print("parametri scelti nei fold:", sorted(set(res["params"].values())))
res["grid"].mean(axis=0).sort_values(ascending=False).round(3).to_frame("spellF1 medio (in-sample)")

## 5. Un tratto di partita: gt vs baseline vs decoder
Tre minuti a partire dal decimo minuto della prima partita DFL. Blu = H, arancione = A, grigio = D. Gli sfarfallii del baseline sono le strisce sottili.

In [ ]:
from matplotlib.colors import ListedColormap
match, gt, preds = decoder.example(res)
a = min(25 * 60 * 10, len(gt) // 4)  # dal 10° minuto (o da 1/4 se la partita è corta)
z = min(a + 25 * 60 * 3, len(gt))
rows = {"ground truth": gt, **preds}
codes = np.array([[{"H": 0, "A": 1, "D": 2}[x] for x in v[a:z]] for v in rows.values()])
fig, ax = plt.subplots(figsize=(14, 2.6))
ax.imshow(codes, aspect="auto", interpolation="nearest", vmin=0, vmax=2,
          cmap=ListedColormap(["#1f77b4", "#ff7f0e", "#bbbbbb"]),
          extent=[a / 25, z / 25, len(rows) - 0.5, -0.5])
ax.set_yticks(range(len(rows)))
ax.set_yticklabels(list(rows))
ax.set_xlabel("secondi")
ax.set_title(match)
plt.tight_layout()
plt.show()

## 6. Output e riferimenti
`<match>_decoded.npz` in `dfl_paths.out_dir()`: `labels` (F,) '<U1' in {H, A, D} e `variant` (parametri, seed, fonti usate). Su Kaggle si scaricano da `/kaggle/working/dfl_outputs`.

* Bassek et al. 2025: dataset DFL (CC BY 4.0).
* Kim et al. 2026, *PathCRF*, KDD: decodifica strutturata con transizioni dinamiche sullo stesso dataset.
* Lang et al. 2022: probabilità di palla morta filtrata con mediana ~3.6 s; interruzioni DFL <= 1 s quasi assenti.
* `EVALUATION.md`, `evaluation/possession_metrics.py`: la metrica.